# BAREC Test Set Prediction and Evaluation

This notebook demonstrates how to:
1. Load a pretrained PIXEL model for BAREC readability classification
2. Make predictions on the test set
3. Save results to CSV format
4. Compute evaluation metrics using the official evaluation script

## Prerequisites
Make sure you have the evaluation script available at the correct path.

In [ ]:
import os
import sys
import logging
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from typing import Dict, List, Optional
import subprocess

# PIXEL specific imports
from pixel import (
    PIXELForSequenceClassification,
    PangoCairoTextRenderer,
    BARECDataset,
    Modality,
    get_transforms,
)
from torch.utils.data import DataLoader

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ All imports successful!")

## Configuration

In [ ]:
# Configuration
CONFIG = {
    # Model settings
    "model_path": "bensapir/pixel-barec-pretrain",  # Change to your model path
    "renderer_path": "Team-PIXEL/pixel-base",  # Base renderer
    
    # Data settings
    "dataset_name": "CAMeL-Lab/BAREC-Shared-Task-2025-sent",  # Sentence-level dataset
    "split": "test",  # Test split
    "max_seq_length": 256,
    "num_labels": 19,
    
    # Prediction settings
    "batch_size": 16,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    
    # Output settings
    "output_csv": "barec_test_predictions.csv",
    "task_type": "Sent",  # "Sent" for sentence-level
    
    # Evaluation script path
    "eval_script_path": "/Users/bensa/Projects/COMPUTATIONAL_LINGUISTICS/barec-shared-task-2025/scripts/eval.py"
}

print(f"🔧 Configuration:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")

## Step 1: Load Model and Renderer

In [ ]:
def load_model_and_renderer(config: Dict) -> tuple:
    """
    Load the pretrained PIXEL model and text renderer.
    """
    print("🤖 Loading model and renderer...")
    
    # Load text renderer
    renderer = PangoCairoTextRenderer.from_pretrained(
        config["renderer_path"],
        rgb=False  # Use grayscale rendering
    )
    renderer.max_seq_length = config["max_seq_length"]
    
    # Load model
    model = PIXELForSequenceClassification.from_pretrained(
        config["model_path"],
        num_labels=config["num_labels"]
    )
    
    # Move to device
    model = model.to(config["device"])
    model.eval()
    
    print(f"   ✅ Model loaded: {config['model_path']}")
    print(f"   ✅ Renderer loaded: {config['renderer_path']}")
    print(f"   📱 Device: {config['device']}")
    print(f"   🔢 Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    return model, renderer

# Load model and renderer
model, renderer = load_model_and_renderer(CONFIG)

## Step 2: Load Test Dataset

In [ ]:
def load_test_dataset(config: Dict, renderer) -> BARECDataset:
    """
    Load the BAREC test dataset.
    """
    print("📊 Loading test dataset...")
    
    # Set up transforms
    transforms = get_transforms(
        do_resize=True,
        size=(renderer.pixels_per_patch, renderer.pixels_per_patch * renderer.max_seq_length),
    )
    
    # Load dataset
    dataset = BARECDataset(
        dataset_name=config["dataset_name"],
        processor=renderer,
        modality=Modality.IMAGE,
        max_seq_length=config["max_seq_length"],
        split=config["split"],
        transforms=transforms
    )
    
    print(f"   ✅ Loaded {len(dataset)} test examples")
    
    return dataset

# Load test dataset
test_dataset = load_test_dataset(CONFIG, renderer)

## Step 3: Create DataLoader

In [ ]:
def create_test_dataloader(dataset: BARECDataset, config: Dict) -> DataLoader:
    """
    Create DataLoader for test predictions.
    """
    def collate_fn(batch):
        """Custom collate function for PIXEL data."""
        pixel_values = torch.stack([item['pixel_values'] for item in batch])
        attention_mask = torch.stack([item['attention_mask'] for item in batch])
        
        # Include labels if available (for verification)
        batch_dict = {
            'pixel_values': pixel_values,
            'attention_mask': attention_mask
        }
        
        if 'label' in batch[0]:
            labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
            batch_dict['labels'] = labels
        
        return batch_dict
    
    dataloader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=False,  # Keep original order for test set
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=True if config["device"] == 'cuda' else False
    )
    
    print(f"📦 Created DataLoader with {len(dataloader)} batches")
    return dataloader

# Create dataloader
test_dataloader = create_test_dataloader(test_dataset, CONFIG)

## Step 4: Make Predictions

In [ ]:
def make_predictions(model, dataloader, device: str) -> tuple:
    """
    Make predictions on the test set.
    """
    print("🔮 Making predictions...")
    
    model.eval()
    all_predictions = []
    all_probabilities = []
    all_labels = []
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc="Predicting")):
            # Move batch to device
            input_batch = {
                'pixel_values': batch['pixel_values'].to(device),
                'attention_mask': batch['attention_mask'].to(device)
            }
            
            # Forward pass
            outputs = model(**input_batch)
            
            # Get predictions and probabilities
            logits = outputs.logits
            probabilities = torch.nn.functional.softmax(logits, dim=-1)
            predictions = torch.argmax(logits, dim=-1)
            
            # Store results
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            
            # Store labels if available
            if 'labels' in batch:
                all_labels.extend(batch['labels'].numpy())
    
    print(f"   ✅ Generated {len(all_predictions)} predictions")
    
    return (
        np.array(all_predictions), 
        np.array(all_probabilities),
        np.array(all_labels) if all_labels else None
    )

# Make predictions
predictions, probabilities, true_labels = make_predictions(model, test_dataloader, CONFIG["device"])

## Step 5: Prepare Results and Save to CSV

In [ ]:
def save_predictions_to_csv(dataset: BARECDataset, predictions: np.ndarray, 
                           probabilities: np.ndarray, config: Dict) -> str:
    """
    Save predictions to CSV file in the required format.
    """
    print("💾 Preparing and saving results to CSV...")
    
    # Get dataset IDs - assuming the dataset has example IDs
    # If your dataset doesn't have IDs, we'll create sequential ones
    try:
        # Try to get IDs from the dataset examples
        example_ids = []
        for i in range(len(dataset)):
            # Check if dataset has ID field
            if hasattr(dataset.examples[i], 'id'):
                example_ids.append(dataset.examples[i].id)
            else:
                # Create sequential IDs
                example_ids.append(f"sent_{i+1}")
    except:
        # Fallback: create sequential IDs
        example_ids = [f"sent_{i+1}" for i in range(len(predictions))]
    
    # Convert 0-based predictions to 1-based (1-19)
    predictions_1_based = predictions + 1
    
    # Calculate confidence scores (max probability)
    confidence_scores = np.max(probabilities, axis=1)
    
    # Prepare DataFrame
    results_data = {
        "Sentence ID": example_ids,
        "Prediction": predictions_1_based.tolist()
    }
    
    # Add optional columns for analysis
    results_data["Confidence"] = confidence_scores.tolist()
    
    # Create DataFrame
    results_df = pd.DataFrame(results_data)
    
    # Save to CSV
    output_path = config["output_csv"]
    results_df.to_csv(output_path, index=False)
    
    print(f"   ✅ Results saved to: {output_path}")
    print(f"   📊 Shape: {results_df.shape}")
    print(f"   🎯 Prediction range: {predictions_1_based.min()} - {predictions_1_based.max()}")
    print(f"   📈 Mean confidence: {confidence_scores.mean():.3f}")
    
    # Display first few rows
    print("\n📋 First 5 predictions:")
    print(results_df.head())
    
    return output_path

# Save predictions to CSV
csv_path = save_predictions_to_csv(test_dataset, predictions, probabilities, CONFIG)

## Step 6: Validation Check (Optional)

Let's verify our CSV format and show some prediction statistics.

In [ ]:
def validate_csv_format(csv_path: str) -> None:
    """
    Validate the CSV format and show statistics.
    """
    print("🔍 Validating CSV format...")
    
    # Load the CSV
    df = pd.read_csv(csv_path)
    
    print(f"   📊 CSV Shape: {df.shape}")
    print(f"   📋 Columns: {df.columns.tolist()}")
    
    # Check required columns
    required_columns = ["Sentence ID", "Prediction"]
    missing_columns = [col for col in required_columns if col not in df.columns]
    
    if missing_columns:
        print(f"   ❌ Missing required columns: {missing_columns}")
    else:
        print(f"   ✅ All required columns present")
    
    # Check prediction values
    predictions = df["Prediction"]
    valid_range = (predictions >= 1) & (predictions <= 19)
    
    print(f"   🎯 Prediction range: {predictions.min()} - {predictions.max()}")
    print(f"   ✅ Valid predictions: {valid_range.sum()}/{len(predictions)}")
    
    if not valid_range.all():
        invalid_preds = predictions[~valid_range]
        print(f"   ⚠️ Invalid predictions found: {invalid_preds.tolist()}")
    
    # Show prediction distribution
    print("\n📈 Prediction Distribution:")
    pred_counts = predictions.value_counts().sort_index()
    for level, count in pred_counts.items():
        percentage = (count / len(predictions)) * 100
        print(f"   Level {level:2d}: {count:4d} ({percentage:5.1f}%)")

# Validate the CSV
validate_csv_format(csv_path)

## Step 7: Run Official Evaluation Script

In [ ]:
def run_evaluation_script(csv_path: str, config: Dict) -> None:
    """
    Run the official evaluation script to compute metrics.
    """
    print("📊 Running official evaluation script...")
    
    eval_script = config["eval_script_path"]
    
    # Check if evaluation script exists
    if not os.path.exists(eval_script):
        print(f"   ❌ Evaluation script not found at: {eval_script}")
        print("   Please update the 'eval_script_path' in CONFIG or provide the correct path.")
        return
    
    # Prepare command
    cmd = [
        "python", eval_script,
        "--output", csv_path,
        "--split", "Test",  # Test split
        "--task", config["task_type"]  # "Sent" for sentence-level
    ]
    
    print(f"   🚀 Running command: {' '.join(cmd)}")
    
    try:
        # Run the evaluation script
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        
        print("   ✅ Evaluation completed successfully!")
        print("\n📊 Evaluation Results:")
        print(result.stdout)
        
        if result.stderr:
            print("\n⚠️ Warnings/Errors:")
            print(result.stderr)
            
    except subprocess.CalledProcessError as e:
        print(f"   ❌ Evaluation script failed with return code {e.returncode}")
        print(f"   Error output: {e.stderr}")
        print(f"   Standard output: {e.stdout}")
    except Exception as e:
        print(f"   ❌ Unexpected error: {e}")

# Run evaluation
run_evaluation_script(csv_path, CONFIG)

## Step 8: Manual Metrics Calculation (Backup)

If the evaluation script doesn't work, we can calculate some basic metrics manually if we have true labels.

In [ ]:
def calculate_manual_metrics(predictions: np.ndarray, true_labels: np.ndarray = None) -> None:
    """
    Calculate metrics manually if true labels are available.
    """
    if true_labels is None:
        print("⚠️ No true labels available for manual metric calculation")
        return
    
    print("🧮 Calculating manual metrics...")
    
    from sklearn.metrics import accuracy_score, cohen_kappa_score, mean_absolute_error
    
    # Convert to 1-based for consistency
    pred_1based = predictions + 1
    true_1based = true_labels + 1
    
    # Calculate metrics
    accuracy = accuracy_score(true_1based, pred_1based)
    qwk = cohen_kappa_score(true_1based, pred_1based, weights='quadratic')
    mae = mean_absolute_error(true_1based, pred_1based)
    
    # Adjacent accuracy (within 1 level)
    adjacent_acc = np.mean(np.abs(true_1based - pred_1based) <= 1)
    
    print(f"\n📊 Manual Metrics (verification):")
    print(f"   Accuracy: {accuracy*100:.4f}%")
    print(f"   Accuracy +/-1: {adjacent_acc*100:.4f}%")
    print(f"   Quadratic Cohen's Kappa: {qwk*100:.4f}%")
    print(f"   Mean Absolute Error: {mae:.6f}")

# Calculate manual metrics if true labels are available
if true_labels is not None:
    calculate_manual_metrics(predictions, true_labels)
else:
    print("ℹ️ True labels not available in test set - this is normal for competition submissions")

## Step 9: Summary and Next Steps

In [ ]:
print("🎉 Prediction and Evaluation Complete!")
print("\n📋 Summary:")
print(f"   Model used: {CONFIG['model_path']}")
print(f"   Dataset: {CONFIG['dataset_name']}")
print(f"   Test examples: {len(predictions)}")
print(f"   Output file: {CONFIG['output_csv']}")
print(f"   Task type: {CONFIG['task_type']} (Sentence-level)")

print("\n📊 Prediction Statistics:")
print(f"   Mean prediction: {(predictions + 1).mean():.2f}")
print(f"   Std prediction: {(predictions + 1).std():.2f}")
print(f"   Min/Max: {(predictions + 1).min()}/{(predictions + 1).max()}")
print(f"   Mean confidence: {probabilities.max(axis=1).mean():.3f}")

print("\n✅ Next Steps:")
print(f"   1. Your predictions are saved in: {CONFIG['output_csv']}")
print(f"   2. The file is ready for submission to the BAREC shared task")
print(f"   3. Official metrics have been calculated using the evaluation script")
print(f"   4. You can analyze the results or adjust your model if needed")

# Show final CSV path for easy access
print(f"\n📁 Final output file: {os.path.abspath(CONFIG['output_csv'])}")